# Машинное обучение, ФКН ВШЭ

# Практическое задание 9. Несбалансированные задачи

## Общая информация
Дата выдачи: 01.03.2026

Мягкий дедлайн: 15.03.2026 23:59 MSK

Жесткий дедлайн: 22.03.2026 23:59 MSK

## Оценивание и штрафы

Каждая из задач имеет определенную «стоимость» (указана в скобках около задачи). Максимально допустимая оценка за работу — 10 баллов.

Сдавать задание после указанного срока сдачи нельзя.

Задание выполняется самостоятельно. «Похожие» решения считаются плагиатом и все задействованные студенты (в том числе те, у кого списали) не могут получить за него больше 0 баллов (подробнее о плагиате см. на странице курса). Если вы нашли решение какого-то из заданий (или его часть) в открытом источнике, необходимо указать ссылку на этот источник в отдельном блоке в конце вашей работы (скорее всего вы будете не единственным, кто это нашел, поэтому чтобы исключить подозрение в плагиате, необходима ссылка на источник).

Неэффективная реализация кода может негативно отразиться на оценке.

## Формат сдачи
Задания сдаются через систему anytask. Посылка должна содержать:
* Ноутбук homework-practice-xx-Username.ipynb

Username — ваша фамилия и имя на латинице именно в таком порядке

## О задании

В этом задании мы разберем основные техники работы в задачах, где один из классов занимает существенно меньшую долю выборки, чем остальные. Для простоты мы обойдемся бинарной задачей, тем не менее, во многом данные методы можно перенести и на задачи с б**о**льшим числом классов. Кроме того, вы получите очередной бесценный опыт исследования случайной библиотеки случайных индусов с нуля.

In [ ]:
import numpy as np
import optuna
import polars as pl
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import average_precision_score
from sklearn.model_selection import train_test_split

# import pandas as pd
# pd.set_option('display.max_rows', 100)
# pd.set_option('display.max_columns', 100)
# import matplotlib.pyplot as plt
# import seaborn as sns
# sns.set(style='whitegrid')
# from tqdm.auto import tqdm


In [ ]:
import plotly.express as px
from plotly.subplots import make_subplots

layout_dict = {
    "margin": {"l": 20, "r": 20, "t": 40, "b": 20},
    "width": 600,
    "height": 400,
    "paper_bgcolor": "LightSteelBlue",
    "title_font_size": 14,
    "xaxis_title_font_size": 12,
    "yaxis_title_font_size": 12,
}

**Задание -1 (1 балл)**. EDA

В качестве данных для нашей работы возьмем выложенный на kaggle датасет транзакций, в котором нужно выискивать мошеннические проводки: [клик](https://www.kaggle.com/mlg-ulb/creditcardfraud). Данная задача по определению подходит под несбалансированную, что можно сказать даже без наличия каких-либо данных (понятно, что среди всех транзакций клиентов очень малая часть будет мошеннической).

Загрузим данные, проведем некоторые классические манипуляции.

In [ ]:
!export KAGGLE_API_TOKEN=$(cat /data/config/.kaggle/token) && /opt/venv/bin/kaggle datasets list -s mlg-ulb/creditcardfraud

In [ ]:
!mkdir -p /data/ml-course-hse/ml2-2026-spring/homework-practice-09-imbalanced && export KAGGLE_API_TOKEN=$(cat /data/config/.kaggle/token) && /opt/venv/bin/kaggle datasets download mlg-ulb/creditcardfraud -p /data/ml-course-hse/ml2-2026-spring/homework-practice-09-imbalanced --unzip

In [ ]:
df = pl.read_csv(
    "/data/ml-course-hse/ml2-2026-spring/homework-practice-09-imbalanced/creditcard.csv",
    infer_schema_length=None,
)
df.sample(5)

In [ ]:
df.describe()

Наши данные были анонимизированы. Мы имеем 30 признаков, из которых 28 - это результаты PCA-преобразования на исходном датасете. Еще 2 признака представляют собой время в секундах, прошедшее с момента первой транзакции в датасете, и размер транзакции. Скажите, какова доля положительных объектов в выборке?

In [ ]:
df["Class"].value_counts()

In [ ]:
df.select(pl.col("Class").mean()).item()

In [ ]:
fig = px.histogram(
    df.drop("Time", "Class", "Amount")
    .unpivot(variable_name="feature", value_name="value")
    .sample(100000)
    .sort("feature"),
    x="value",
    facet_col="feature",
    facet_col_wrap=4,
    nbins=50,
    facet_row_spacing=0.03,
    facet_col_spacing=0.04,
)
fig.update_layout(**layout_dict)
fig.update_layout(width=1800, height=3000)

fig.update_traces(bingroup=None)
fig.update_xaxes(matches=None, showticklabels=True)
fig.update_yaxes(matches=None, showticklabels=True)

fig.show()

Начнем с обработки времени. Секунды сами по себе не несут большой информации о зависимостях в данных, попробуйте по ним создать признаки "час" (от 0 до 23) и "день" (от 0 до ...) в аналогичной манере (принимая первый объект выборки за начальную точку). Сколько дней покрывают данные?

In [ ]:
df = df.with_columns(
    hour=pl.col("Time").cast(pl.Int64).mod(3600 * 24).floordiv(3600),
    day=pl.col("Time").cast(pl.Int64).floordiv(3600 * 24),
)

Постройте следующие графики:

1. Распределение числа транзакций по каждому часу (line-plot).
2. Распределение доли мошеннических транзакций по каждому часу (line-plot)
3. То же самое для дней (здесь можно использовать bar-plot, так как дней должно быть немного).

Какие выводы можно сделать из графиков? На ваш взгляд, как можно связать полученные нами часы с реальными часами в сутках?

In [ ]:
fig1 = px.line(
    df.group_by("hour", "day")
    .agg(count=pl.len())
    .group_by("hour")
    .agg(pl.col("count").mean())
    .sort("hour"),
    x="hour",
    y="count",
)

fig2 = px.line(
    df.group_by("hour").agg(fraud_avg=pl.col("Class").mean()).sort("hour"),
    x="hour",
    y="fraud_avg",
)

fig3 = px.bar(
    df.group_by("day").agg(count=pl.len()).sort("day"),
    x="day",
    y="count",
)

fig4 = px.bar(
    df.group_by("day").agg(fraud_avg=pl.col("Class").mean()).sort("day"),
    x="day",
    y="fraud_avg",
)


fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=(
        "Transaction count by hour",
        "Average number of fraud transactions by hour",
        "Transaction count by day",
        "Average number of fraud transactions by day",
    ),
    horizontal_spacing=0.05,
    vertical_spacing=0.12,
)


for trace in fig1.data:
    fig.add_trace(trace.update(showlegend=False), row=1, col=1)

for trace in fig2.data:
    fig.add_trace(trace.update(showlegend=False), row=1, col=2)

for trace in fig3.data:
    fig.add_trace(trace.update(showlegend=False), row=2, col=1)

for trace in fig4.data:
    fig.add_trace(trace.update(showlegend=False), row=2, col=2)


fig.update_layout(**layout_dict)
fig.update_layout(width=1200, height=800)


С анонимизированными признаками вряд ли можно придумать что-то интересное. Попробуйте (например, с помощью корреляции?) выбрать несколько наиболее важных признаков и поглядеть на различия в их распределении для разных классов.

In [ ]:
# feature_name = df.select(pl.corr(v, pl.col("Class")).abs() for v in df.columns).transpose(
#     include_header=True, header_name="feature_name", column_names=["abs_correlation"]
# ).sort("abs_correlation", descending=True)[1, 'feature_name']

feature_name = (
    df.drop("Time", "day", "hour", "Amount")
    .unpivot(index="Class", variable_name="feature_name", value_name="value")
    .group_by("feature_name", "Class")
    .agg(avg=pl.col("value").mean(), var=pl.col("value").var(ddof=1), count=pl.len())
    .sort("feature_name", "Class")
    .group_by("feature_name")
    .agg(
        t_stat=(pl.col("avg").last() - pl.col("avg").first())
        / (
            (
                pl.col("var").first() / pl.col("count").first()
                + pl.col("var").last() / pl.col("count").last()
            )
            ** 0.5
        )
    )
    .sort("t_stat")[-9, "feature_name"]
)

fig = px.histogram(
    df.select(
        pl.col(feature_name).clip(
            pl.col(feature_name).quantile(0.0005), pl.col(feature_name).quantile(0.9995)
        ),
        pl.col("Class"),
    ),
    x=feature_name,
    color="Class",
    nbins=100,
)

fig.update_layout(**layout_dict)
fig.update_layout(title=feature_name)

Теперь давайте разделим данные. Отделите хронологически последние 20% транзакций и поделите их пополам (также хронологически, т.е. без перемешивания) на валидационные и тестовые. Это разбиение не совсем корректно (как можно было заметить, мошеннические транзакции имеют разное распределение во времени - по-хорошему, нам стоило бы выделить целые сутки записей как под валидацию, так и под тест), тем не менее, мы не сможем получить больше данных для адекватного контроля, поэтому обойдемся этим. 

In [ ]:
df = df.sort("Time")

df_train, df_val = train_test_split(df, train_size=0.8, shuffle=False)
df_val, df_test = train_test_split(df_val, train_size=0.5, shuffle=False)

print(f"Train samples: {len(df_train)}")
print(f"Validation samples: {len(df_val)}")
print(f"Test samples: {len(df_test)}")

In [ ]:
del df

# Часть 1. Несбалансированная классификация.

**Задание 0. (1 балл)**: перед началом работы давайте поговорим о том, как мы будем оценивать качество. Классические метрики для качества классификации чаще всего "ломаются" на задачах с сильным перекосом. Чему будет равно значение accuracy для наивного предсказания (= мажорный класс для каждого объекта)? (можете не отвечать, просто подумайте)

Из курса МО-1 вам уже известно, что мы можем использовать в таких задачах `AUC-PR` и получать адекватные показатели. Можно сказать, что `AUC-PR` представляет собой матожидание `precision` по распределению, заданному выигрышем в `recall` при смене порога. 

Тем не менее, существуют и другие, не менее интересные метрики. Одной из таких метрик является коэффициент Каппа Коэна, представляющий собой нормализованную `accuracy`:

$$\kappa = \frac{p_o - p_e}{1 - p_e}$$

Данная метрика служит в качестве меры согласованности между двумя независимыми предсказателями, но ничего не знает про "верные" и "предсказанные" метки (в отличие от многих других метрик машинного обучения). Здесь $p_o$ - доля согласованных предсказаний, а $p_e$ - доля согласованных предсказаний, которая могла бы получиться при случайных ответах предсказателей. В нашем случае это работает так:

• В качестве $p_o$ берем accuracy

• В качестве $p_e$ примем следующую величину - вероятность случайного соглашения позитивных ответов (произведение долей позитивных ответов в обоих предсказаниях) плюс вероятность случайного соглашения негативных ответов (произведение долей негативных ответов в обоих предсказаниях)

Метрика принимает значения от -1 до 1, где 1 - полная согласованность, 0 - согласованность на уровне рандома, -1 - совсем плохо. Как уже говорилось, метрика не различает "верные" и "предсказанные" метки, поэтому является симметричной (можете использовать это для отладки):

In [ ]:
def get_cohen_kappa(y, y_hat):
    po = (y == y_hat).mean()
    pe = y.mean() * y_hat.mean() + (1.0 - y).mean() * (1.0 - y_hat).mean()

    return (po - pe) / (1.0 - pe)

In [ ]:
assert 1 == get_cohen_kappa(df_train["Class"], df_train["Class"])
print(
    get_cohen_kappa(
        df_train["Class"],
        pl.Series(
            np.random.binomial(n=1, p=df_train["Class"].mean(), size=len(df_train))
        ),
    )
)

Еще одной метрикой в такой задаче служит коэффициент корреляции Мэтьюза, выражающийся в терминах матрицы ошибок следующим образом:

$$\text{MCC} = \frac{TP\times TN - FP \times FN}{\sqrt{(TP+FP)(TP+FN)(TN+FP)(TN+FN)}}$$

Метрика принимает значения от -1 до 1, интерпретируемые аналогичным образом. 

In [ ]:
# Note: this is simply the Pearson correlation between y and y_hat
def get_matthews_corrcoef(y, y_hat):
    err_matrix = (
        pl.DataFrame({"y": y, "y_hat": y_hat})
        .pivot(
            on="y",
            index="y_hat",
            values="y_hat",
            aggregate_function=pl.len(),
            sort_columns=True,
        )
        .sort("y_hat")
    )
    if pl.Series(y).n_unique() < 2 or pl.Series(y_hat).n_unique() < 2:
        return 0.0

    # note that the order is ill-defined, but the output is correct regardless
    tp = err_matrix[1, 2]
    tn = err_matrix[0, 1]
    fp = err_matrix[1, 1]
    fn = err_matrix[0, 2]

    return (tp * tn - fp * fn) / ((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn)) ** 0.5

In [ ]:
get_matthews_corrcoef(
    df_train["Class"],
    pl.Series(np.random.binomial(n=1, p=df_train["Class"].mean(), size=len(df_train))),
)

Обратите внимание, что эти метрики вычисляются на бинаризованных предсказаниях, поэтому может иметь смысл дополнительная настройка порога бинаризации.

Давайте проверим, что наши метрики действительно подходят под задачу. Вычислите их значения для наивного предсказания (aka мажорный класс для всех объектов):

In [ ]:
get_cohen_kappa(
    df_train["Class"],
    pl.repeat(df_train["Class"].mode().item(), len(df_train), eager=True),
)

In [ ]:
get_matthews_corrcoef(
    df_train["Class"],
    pl.repeat(df_train["Class"].mode().item(), len(df_train), eager=True),
)

Давайте запустим бейзлайн-решение для нашей задачи. С чего же начнем? Возьмите `catboost` и обучите его классификатор на наших данных (используйте все признаки). Вычислите значения всех метрик на тестовой части, для контроля переобучения используйте валидационную (здесь и далее везде, где фигурирует `catboost`). 

In [ ]:
train_pool = Pool(df_train.select(pl.selectors.starts_with("V")), df_train["Class"])
val_pool = Pool(df_val.select(pl.selectors.starts_with("V")), df_val["Class"])
test_pool = Pool(df_test.select(pl.selectors.starts_with("V")), df_test["Class"])

In [ ]:
cbc = CatBoostClassifier(
    iterations=1000,  # default is 1000
    loss_function="Logloss",
    learning_rate=1e-1,
    early_stopping_rounds=50,  # rounds without improvement
    max_depth=5,
    min_data_in_leaf=None,
    rsm=None,  # rate of subsampling tree features
    bootstrap_type=None,
    subsample=None,
    l2_leaf_reg=None,
    model_size_reg=None,
    random_seed=123,
)

# subsample bootstrap_type class_weights

Если вы все сделали правильно, у вас должны были получиться значения в районе 0.7.

In [ ]:
cbc.fit(train_pool, eval_set=val_pool)

In [ ]:
cbc.predict(data=df_train.select(pl.selectors.starts_with("V"))).sum()

In [ ]:
def tree_config_objective(trial: optuna.Trial) -> float:
    max_depth = trial.suggest_int("max_depth", 3, 10)
    min_data_in_leaf = trial.suggest_int("min_data_in_leaf", 3, 8)
    rsm = trial.suggest_float("rsm", 0.1, 1.0)

    cbc_tmp = CatBoostClassifier(
        # early_stopping_rounds=50, # rounds without improvement
        iterations=200,
        loss_function="Logloss",
        learning_rate=1e-1,
        max_depth=max_depth,
        min_data_in_leaf=min_data_in_leaf,
        rsm=rsm,
        bootstrap_type=None,
        subsample=None,
        l2_leaf_reg=None,
        model_size_reg=None,
        random_seed=123,
    )

    cbc_tmp.fit(
        train_pool,
        eval_set=val_pool,
        verbose=False,
    )

    proba = cbc_tmp.predict_proba(df_val.select(pl.selectors.starts_with("V")))[:, 1]
    pred = pl.Series((proba >= 0.5).astype(int))

    trial.set_user_attr(
        "log_loss",
        cbc_tmp.get_evals_result()["validation"]["Logloss"][
            cbc_tmp.get_best_iteration()
        ],
    )
    trial.set_user_attr("cohen_kappa", get_cohen_kappa(df_val["Class"], pred))
    trial.set_user_attr(
        "matthews_correlation", get_matthews_corrcoef(df_val["Class"], pred)
    )

    return average_precision_score(df_val["Class"], proba)


In [ ]:
study_name = "tree_config_null"

sampler = optuna.samplers.TPESampler(
    seed=10
)  # Make the sampler behave in a deterministic way.
study1 = optuna.create_study(
    study_name=study_name,
    sampler=sampler,
    pruner=optuna.pruners.HyperbandPruner(),
    # pruner=optuna.pruners.PatientPruner(optuna.pruners.MedianPruner(), patience=5),
    # storage=f"sqlite:////data/{study_name}.db",
    direction="maximize",
    # load_if_exists=True,
)
study1.optimize(
    tree_config_objective,
    n_trials=40,
)
print(study1.best_params)

In [ ]:
study1.trials_dataframe()

In [ ]:
study1.best_params  # = {'max_depth': 7, 'min_data_in_leaf': 6, 'rsm': 0.5138945150190546}

In [ ]:
def bootstrap_objective(trial: optuna.Trial) -> float:
    # bootstrap_type = 'Bernoulli'
    bootstrap_type = trial.suggest_categorical("bootstrap_type", ("Bernoulli", "MVS"))
    subsample = (
        trial.suggest_float("subsample", 0.1, 1.0)
        if bootstrap_type == "Bernoulli"
        else None
    )

    cbc_tmp = CatBoostClassifier(
        # early_stopping_rounds=50, # rounds without improvement
        iterations=200,
        loss_function="Logloss",
        learning_rate=1e-1,
        # max_depth=max_depth,
        # min_data_in_leaf=min_samples_leaf,
        # rsm=rsm,
        bootstrap_type=bootstrap_type,
        subsample=subsample,
        l2_leaf_reg=None,
        model_size_reg=None,
        random_seed=123,
        # **{"max_depth": 7, "min_data_in_leaf": 6, "rsm": 0.5138945150190546},
        **study1.best_params,
    )

    cbc_tmp.fit(
        train_pool,
        eval_set=val_pool,
        verbose=False,
    )

    proba = cbc_tmp.predict_proba(df_val.select(pl.selectors.starts_with("V")))[:, 1]
    pred = pl.Series((proba >= 0.5).astype(int))

    trial.set_user_attr(
        "log_loss",
        cbc_tmp.get_evals_result()["validation"]["Logloss"][
            cbc_tmp.get_best_iteration()
        ],
    )
    trial.set_user_attr("cohen_kappa", get_cohen_kappa(df_val["Class"], pred))
    trial.set_user_attr(
        "matthews_correlation", get_matthews_corrcoef(df_val["Class"], pred)
    )

    return average_precision_score(df_val["Class"], proba)


In [ ]:
study_name = "bootstrap_null"

sampler = optuna.samplers.TPESampler(
    seed=10
)  # Make the sampler behave in a deterministic way.
study2 = optuna.create_study(
    study_name=study_name,
    sampler=sampler,
    pruner=optuna.pruners.HyperbandPruner(),
    # pruner=optuna.pruners.PatientPruner(optuna.pruners.MedianPruner(), patience=5),
    # storage=f"sqlite:////data/{study_name}.db",
    direction="maximize",
    # load_if_exists=True,
)
study2.optimize(
    bootstrap_objective,
    n_trials=20,
)
print(study2.best_params)

In [ ]:
def regularization_objective(trial: optuna.Trial) -> float:
    l2_leaf_reg = trial.suggest_float("l2_leaf_reg", 1e-1, 1e3, log=True)
    model_size_reg = trial.suggest_float("model_size_reg", 1e-1, 1e3, log=True)

    cbc_tmp = CatBoostClassifier(
        # early_stopping_rounds=50, # rounds without improvement
        iterations=200,
        loss_function="Logloss",
        learning_rate=1e-1,
        # max_depth=max_depth,
        # min_data_in_leaf=min_samples_leaf,
        # rsm=rsm,
        # bootstrap_type=bootstrap_type,
        # subsample=subsample,
        l2_leaf_reg=l2_leaf_reg,
        model_size_reg=model_size_reg,
        random_seed=123,
        # **{"max_depth": 7, "min_data_in_leaf": 6, "rsm": 0.5138945150190546},
        **study1.best_params,
        # **{"bootstrap_type": "MVS"},
        **study2.best_params,
    )

    cbc_tmp.fit(
        train_pool,
        eval_set=val_pool,
        verbose=False,
    )

    proba = cbc_tmp.predict_proba(df_val.select(pl.selectors.starts_with("V")))[:, 1]
    pred = pl.Series((proba >= 0.5).astype(int))

    trial.set_user_attr(
        "log_loss",
        cbc_tmp.get_evals_result()["validation"]["Logloss"][
            cbc_tmp.get_best_iteration()
        ],
    )
    trial.set_user_attr("cohen_kappa", get_cohen_kappa(df_val["Class"], pred))
    trial.set_user_attr(
        "matthews_correlation", get_matthews_corrcoef(df_val["Class"], pred)
    )

    return average_precision_score(df_val["Class"], proba)


In [ ]:
study_name = "regularization_null"

sampler = optuna.samplers.TPESampler(
    seed=10
)  # Make the sampler behave in a deterministic way.
study3 = optuna.create_study(
    study_name=study_name,
    sampler=sampler,
    pruner=optuna.pruners.HyperbandPruner(),
    # pruner=optuna.pruners.PatientPruner(optuna.pruners.MedianPruner(), patience=5),
    # storage=f"sqlite:////data/{study_name}.db",
    direction="maximize",
    # load_if_exists=True,
)
study3.optimize(
    regularization_objective,
    n_trials=40,
)
print(study3.best_params)

In [ ]:
iter_scale_times = 10

cbc = CatBoostClassifier(
    # early_stopping_rounds=50, # rounds without improvement
    iterations=200 * iter_scale_times,
    loss_function="Logloss",
    learning_rate=1e-1 / iter_scale_times,
    # max_depth=max_depth,
    # min_data_in_leaf=min_samples_leaf,
    # rsm=rsm,
    # bootstrap_type=bootstrap_type,
    # subsample=subsample,
    # l2_leaf_reg=l2_leaf_reg,
    # model_size_reg=model_size_reg,
    random_seed=123,
    # **{"max_depth": 7, "min_data_in_leaf": 6, "rsm": 0.5138945150190546},
    **study1.best_params,
    # **{"bootstrap_type": "MVS"},
    **study2.best_params,
    # **{"l2_leaf_reg": 2.362559310733596, "model_size_reg": 0.1143381070429593},
    **study3.best_params,
)

In [ ]:
cbc.fit(train_pool, eval_set=val_pool)

In [ ]:
def get_thresh(y, p_hat, step_size=1e-2):
    thresholds = np.arange(0, 1, step_size)
    return thresholds[
        pl.Series(get_cohen_kappa(y, p_hat >= t) for t in thresholds).arg_max()
    ]

In [ ]:
def print_stats(df_val, df_test, model, threshold=None):
    proba = model.predict_proba(df_val.select(pl.selectors.starts_with("V")))[:, 1]
    proba_test = model.predict_proba(df_test.select(pl.selectors.starts_with("V")))[
        :, 1
    ]

    if threshold is None:
        threshold = get_thresh(df_val["Class"], proba)

    pred = pl.Series((proba >= threshold).astype(int))
    pred_test = pl.Series((proba_test >= threshold).astype(int))

    res = {
        "val": {
            "auc_pr": average_precision_score(df_val["Class"], proba),
            "cohen_kappa": get_cohen_kappa(df_val["Class"], pred),
            "matthews_corr": get_matthews_corrcoef(df_val["Class"], pred),
        },
        "test": {
            "auc_pr": average_precision_score(df_test["Class"], proba_test),
            "cohen_kappa": get_cohen_kappa(df_test["Class"], pred_test),
            "matthews_corr": get_matthews_corrcoef(df_test["Class"], pred_test),
        },
    }

    print(f"Threshold {threshold}")
    print(
        f"Cohen Kappa val: {res['val']['cohen_kappa']:.3f}, test: {res['test']['cohen_kappa']:.3f}"
    )
    print(
        f"Matthews correlation val: {res['val']['matthews_corr']:.3f}, test: {res['test']['matthews_corr']:.3f}"
    )

    print(f"AUC-PR val: {res['val']['auc_pr']:.3f}, test: {res['test']['auc_pr']:.3f}")
    return res

In [ ]:
proba = cbc.predict_proba(df_val.select(pl.selectors.starts_with("V")))[:, 1]
pred = pl.Series((proba >= 0.5).astype(int))

proba_test = cbc.predict_proba(df_test.select(pl.selectors.starts_with("V")))[:, 1]
pred_test = pl.Series((proba_test >= 0.5).astype(int))

print(
    f"Cohen Kappa val: {get_cohen_kappa(df_val['Class'], pred):.3f}, test: {get_cohen_kappa(df_test['Class'], pred_test):.3f}"
)
print(
    f"Matthews correlation val: {get_matthews_corrcoef(df_val['Class'], pred):.3f}, test: {get_matthews_corrcoef(df_test['Class'], pred_test):.3f}"
)

print(
    f"AUC-PR val: {average_precision_score(df_val['Class'], proba):.3f}, test: {average_precision_score(df_test['Class'], proba_test):.3f}"
)

In [ ]:
threshold = get_thresh(df_val["Class"], proba)

pred = pl.Series((proba >= threshold).astype(int))
pred_test = pl.Series((proba_test >= threshold).astype(int))

print(f"Threshold {threshold}")
print(
    f"Cohen Kappa val: {get_cohen_kappa(df_val['Class'], pred):.3f}, test: {get_cohen_kappa(df_test['Class'], pred_test):.3f}"
)
print(
    f"Matthews correlation val: {get_matthews_corrcoef(df_val['Class'], pred):.3f}, test: {get_matthews_corrcoef(df_test['Class'], pred_test):.3f}"
)

print(
    f"AUC-PR val: {average_precision_score(df_val['Class'], proba):.3f}, test: {average_precision_score(df_test['Class'], proba_test):.3f}"
)

**Задание 1. (1 балл)**. Многие реализации методов предлагают встроенные способы для борьбы с нашей проблемой. Самое часто встречающееся решение - просто добавить вес в функции потерь для минорного класса (таким образом, ошибка на объекте минорного класса будет весить больше, чем для мажорного). В `catboost` это также реализовано, причем для бинарной задачи это можно сделать целыми двумя способами (можете выбрать любой, на свой вкус, автор задания предпочитает отдельный скейлинг для минорного класса). Чаще всего в качестве веса берется отношение числа объектов мажорного класса к числу минорного. Попробуйте обучить модель с таким скалированием и сравните метрики на тестовой части с бейзлайном.

In [ ]:
cbc_wts = CatBoostClassifier(
    # early_stopping_rounds=50, # rounds without improvement
    iterations=200 * iter_scale_times,
    loss_function="Logloss",
    learning_rate=1e-1 / iter_scale_times,
    # max_depth=max_depth,
    # min_data_in_leaf=min_samples_leaf,
    # rsm=rsm,
    # bootstrap_type=bootstrap_type,
    # subsample=subsample,
    # l2_leaf_reg=l2_leaf_reg,
    # model_size_reg=model_size_reg,
    scale_pos_weight=(lambda x: (1 - x) / x)(df_train["Class"].mean()),
    random_seed=123,
    # **{"max_depth": 7, "min_data_in_leaf": 6, "rsm": 0.5138945150190546},
    **study1.best_params,
    # **{"bootstrap_type": "MVS"},
    **study2.best_params,
    # **{"l2_leaf_reg": 2.362559310733596, "model_size_reg": 0.1143381070429593},
    **study3.best_params,
)

In [ ]:
cbc_wts.fit(train_pool, eval_set=val_pool)

In [ ]:
proba_wts = cbc_wts.predict_proba(df_val.select(pl.selectors.starts_with("V")))[:, 1]
pred_wts = pl.Series((proba_wts >= 0.5).astype(int))

proba_wts_test = cbc_wts.predict_proba(df_test.select(pl.selectors.starts_with("V")))[
    :, 1
]
pred_wts_test = pl.Series((proba_wts_test >= 0.5).astype(int))

print(
    f"Cohen Kappa val: {get_cohen_kappa(df_val['Class'], pred_wts):.3f}, test: {get_cohen_kappa(df_test['Class'], pred_wts_test):.3f}"
)
print(
    f"Matthews correlation val: {get_matthews_corrcoef(df_val['Class'], pred_wts):.3f}, test: {get_matthews_corrcoef(df_test['Class'], pred_wts_test):.3f}"
)

print(
    f"AUC-PR val: {average_precision_score(df_val['Class'], proba_wts):.3f}, test: {average_precision_score(df_test['Class'], proba_wts_test):.3f}"
)

In [ ]:
threshold_wts = get_thresh(df_val["Class"], proba_wts)

pred_wts = pl.Series((proba_wts >= threshold_wts).astype(int))
pred_wts_test = pl.Series((proba_wts_test >= threshold_wts).astype(int))

print(f"Threshold {threshold_wts}")
print(
    f"Cohen Kappa val: {get_cohen_kappa(df_val['Class'], pred_wts):.3f}, test: {get_cohen_kappa(df_test['Class'], pred_wts_test):.3f}"
)
print(
    f"Matthews correlation val: {get_matthews_corrcoef(df_val['Class'], pred_wts):.3f}, test: {get_matthews_corrcoef(df_test['Class'], pred_wts_test):.3f}"
)

print(
    f"AUC-PR val: {average_precision_score(df_val['Class'], proba_wts):.3f}, test: {average_precision_score(df_test['Class'], proba_wts_test):.3f}"
)

Поскольку данный вес будет являться гиперпараметром метода, было бы опрометчиво остановиться на одном значении (тем более, с большой вероятностью у вас все сломалось). Запустите перебор для этого гиперпараметра на валидационной выборке (используйте `PR-AUC`), подберите оптимальный порог бинаризации для $\kappa$ или $\text{MCC}$. Для лучшего найденного веса и порога вычислите все метрики на тестовой части. 

При этом можете также проверить отдельное скалирование в большую сторону для мажорного класса (т.е. веса минорного сделать меньше 1) и экстремальные скалирования (т.е. веса минорного больше, чем в начале этого задания). Какой вес получился оптимальным?

In [ ]:
def weighting_objective(trial: optuna.Trial) -> float:
    scale_pos_weight = trial.suggest_float("scale_pos_weight", 1e-2, 1e4, log=True)

    cbc_tmp = CatBoostClassifier(
        # early_stopping_rounds=50, # rounds without improvement
        iterations=200,
        loss_function="Logloss",
        learning_rate=1e-1,
        # max_depth=max_depth,
        # min_data_in_leaf=min_samples_leaf,
        # rsm=rsm,
        # bootstrap_type=bootstrap_type,
        # subsample=subsample,
        # l2_leaf_reg=l2_leaf_reg,
        # model_size_reg=model_size_reg,
        scale_pos_weight=scale_pos_weight,
        random_seed=123,
        # **{"max_depth": 7, "min_data_in_leaf": 6, "rsm": 0.5138945150190546},
        **study1.best_params,
        # **{"bootstrap_type": "MVS"},
        **study2.best_params,
        # **{"l2_leaf_reg": 2.362559310733596, "model_size_reg": 0.1143381070429593},
        **study3.best_params,
    )

    cbc_tmp.fit(
        train_pool,
        eval_set=val_pool,
        verbose=False,
    )

    proba = cbc_tmp.predict_proba(df_val.select(pl.selectors.starts_with("V")))[:, 1]
    pred = pl.Series((proba >= 0.5).astype(int))

    trial.set_user_attr(
        "log_loss",
        cbc_tmp.get_evals_result()["validation"]["Logloss"][
            cbc_tmp.get_best_iteration()
        ],
    )
    trial.set_user_attr("cohen_kappa", get_cohen_kappa(df_val["Class"], pred))
    trial.set_user_attr(
        "matthews_correlation", get_matthews_corrcoef(df_val["Class"], pred)
    )

    return average_precision_score(df_val["Class"], proba)

In [ ]:
study_name = "weighting_null"

sampler = optuna.samplers.TPESampler(
    seed=10
)  # Make the sampler behave in a deterministic way.
study4 = optuna.create_study(
    study_name=study_name,
    sampler=sampler,
    pruner=optuna.pruners.HyperbandPruner(),
    # pruner=optuna.pruners.PatientPruner(optuna.pruners.MedianPruner(), patience=5),
    # storage=f"sqlite:////data/{study_name}.db",
    direction="maximize",
    # load_if_exists=True,
)
study4.optimize(
    weighting_objective,
    n_trials=40,
)
print(study4.best_params)

In [ ]:
cbc_wts = CatBoostClassifier(
    # early_stopping_rounds=50, # rounds without improvement
    iterations=200 * iter_scale_times,
    loss_function="Logloss",
    learning_rate=1e-1 / iter_scale_times,
    # max_depth=max_depth,
    # min_data_in_leaf=min_samples_leaf,
    # rsm=rsm,
    # bootstrap_type=bootstrap_type,
    # subsample=subsample,
    # l2_leaf_reg=l2_leaf_reg,
    # model_size_reg=model_size_reg,
    random_seed=123,
    # **{"max_depth": 7, "min_data_in_leaf": 6, "rsm": 0.5138945150190546},
    **study1.best_params,
    # **{"bootstrap_type": "MVS"},
    **study2.best_params,
    # **{"l2_leaf_reg": 2.362559310733596, "model_size_reg": 0.1143381070429593},
    **study3.best_params,
    # **{'scale_pos_weight':0.9267767114548491},
    **study4.best_params,
)

In [ ]:
cbc_wts.fit(train_pool, eval_set=val_pool)

In [ ]:
print_stats(df_val, df_test, cbc_wts, threshold=0.5)

In [ ]:
print_stats(df_val, df_test, cbc_wts)

**Задание 2. (1 балл)**. На самом деле, то, что мы сейчас делали, очень схоже с другой распространенной техникой - оверсэмплингом. Фактически, мы можем продублировать все объекты минорного класса и получить тот же эффект, какой был бы при использовании веса, равного 2. Тем не менее, такой подход - это лишь малая часть того, что мы можем проделать с целью повысить число объектов минорного класса. 

Для продолжения работы установим библиотеку [imbalanced-learn](https://imbalanced-learn.org/stable/):

In [ ]:
# !pip3 install imbalanced-learn
# conda install -c conda-forge imbalanced-learn

Первый метод, которым мы воспользуемся, называется SMOTE (его вы уже разбирали на лекции). Кратко напомним суть: мы выбираем случайного кандидата среди $k$ ближайших соседей объекта минорного класса, затем берем точку на отрезке между двумя объектами (т.е. выпуклую комбинацию со случайными коэффициентами) и добавляем в выборку. 

In [ ]:
from imblearn.over_sampling import SMOTE

Используйте SMOTE для ресэмплинга обучающей выборки, на новой выборке обучите модель (вес положительных объектов скалировать не нужно). Замерьте качество на тестовой выборке (**важно!** не преобразовывайте валидационную и тестовую выборку никак - мы не хотим отслеживать качество на объектах, которых в реальности не существует). Сравните полное выравнивание выборки с частичным (т.е. таким, что баланс классов улучшается, но не достигает равенства - скажем, 1:2 и 1:10).

In [ ]:
iter_scale_times = 10

cbc_ovs = CatBoostClassifier(
    # early_stopping_rounds=50, # rounds without improvement
    iterations=200 * iter_scale_times,
    loss_function="Logloss",
    learning_rate=1e-1 / iter_scale_times,
    # max_depth=max_depth,
    # min_data_in_leaf=min_samples_leaf,
    # rsm=rsm,
    # bootstrap_type=bootstrap_type,
    # subsample=subsample,
    # l2_leaf_reg=l2_leaf_reg,
    # model_size_reg=model_size_reg,
    random_seed=123,
    **{"max_depth": 7, "min_data_in_leaf": 6, "rsm": 0.5138945150190546},
    # **study1.best_params,
    **{"bootstrap_type": "MVS"},
    # **study2.best_params,
    **{"l2_leaf_reg": 2.362559310733596, "model_size_reg": 0.1143381070429593},
    # **study3.best_params,
    # **{'scale_pos_weight':0.9267767114548491},
    # **study4.best_params,
)

In [ ]:
ovs_results = pl.DataFrame(
    schema={
        "name": pl.String,
        "val_auc_pr": pl.Float64,
        "test_auc_pr": pl.Float64,
        "val_cohen_kappa": pl.Float64,
        "test_cohen_kappa": pl.Float64,
        "val_matthews_corr": pl.Float64,
        "test_matthews_corr": pl.Float64,
    }
)

In [ ]:
sm = SMOTE(
    sampling_strategy=0.5,
    random_state=132,
)
X_train_ovs, y_train_ovs = sm.fit_resample(
    df_train.to_numpy(), df_train["Class"].to_numpy()
)
X_train_ovs = pl.DataFrame(X_train_ovs, schema=df_train.columns)
y_train_ovs = pl.Series("Class", y_train_ovs)

train_pool_ovs = Pool(X_train_ovs.select(pl.selectors.starts_with("V")), y_train_ovs)
X_train_ovs["Class"].value_counts()

In [ ]:
cbc_ovs.fit(train_pool_ovs, eval_set=val_pool, verbose=False)

ovs_results = ovs_results.vstack(
    pl.DataFrame(
        [
            {
                f"{k0}_{k}": v
                for k0, v0 in print_stats(df_val, df_test, cbc_ovs).items()
                for k, v in v0.items()
            }
        ],
        schema=ovs_results.schema,
    ).with_columns(name=pl.lit("smote_ovs_0.5"))
)

In [ ]:
sm.set_params(sampling_strategy=0.1)
X_train_ovs, y_train_ovs = sm.fit_resample(
    df_train.to_numpy(), df_train["Class"].to_numpy()
)
X_train_ovs = pl.DataFrame(X_train_ovs, schema=df_train.columns)
y_train_ovs = pl.Series("Class", y_train_ovs)

train_pool_ovs = Pool(X_train_ovs.select(pl.selectors.starts_with("V")), y_train_ovs)
X_train_ovs["Class"].value_counts()

In [ ]:
cbc_ovs.fit(train_pool_ovs, eval_set=val_pool, verbose=False)

ovs_results = ovs_results.vstack(
    pl.DataFrame(
        [
            {
                f"{k0}_{k}": v
                for k0, v0 in print_stats(df_val, df_test, cbc_ovs).items()
                for k, v in v0.items()
            }
        ],
        schema=ovs_results.schema,
    ).with_columns(name=pl.lit("smote_ovs_0.1"))
)

**Бонус (1.5 балла)**. Для vanilla SMOTE существуют некоторые модификации, часть из которых была реализована в библиотеке imblearn. Во время лекций/семинаров эти модификации не рассматривались, тем не менее, мы все равно их исследуем.

Найдите статьи о следующих методах и попробуйте вкратце сформулировать, в чем их основная идея (сделайте так, чтобы человек, знакомый с машинным обучением в целом, но не слышавший конкретно про это смог понять):

BorderlineSMOTE - 

SVM-SMOTE - 

K-Means-SMOTE - 

ADASYN - 

Теперь попробуйте сравнить качество всех методов на наших данных.

In [ ]:
from imblearn.over_sampling import ADASYN, SVMSMOTE, BorderlineSMOTE, KMeansSMOTE

In [ ]:
oversample_dict = {
    "svm_smote": SVMSMOTE,
    "adasyn": ADASYN,
    "borderline_smote": BorderlineSMOTE,
    "k_means_smote": KMeansSMOTE,
}

for name, oversampler in oversample_dict.items():
    sm = oversampler(
        random_state=132,
    )
    for sampling_strategy in (0.5, 0.1):
        print(f"{name} with sampling_strategy={sampling_strategy}")
        sm.set_params(sampling_strategy=sampling_strategy)
        if name == "k_means_smote":
            sm.set_params(cluster_balance_threshold=0.005)
        X_train_ovs, y_train_ovs = sm.fit_resample(
            df_train.to_numpy(), df_train["Class"].to_numpy()
        )
        X_train_ovs = pl.DataFrame(X_train_ovs, schema=df_train.columns)
        y_train_ovs = pl.Series("Class", y_train_ovs)

        train_pool_ovs = Pool(
            X_train_ovs.select(pl.selectors.starts_with("V")), y_train_ovs
        )

        cbc_ovs.fit(train_pool_ovs, eval_set=val_pool, verbose=False)

        ovs_results = ovs_results.vstack(
            pl.DataFrame(
                [
                    {
                        f"{k0}_{k}": v
                        for k0, v0 in print_stats(df_val, df_test, cbc_ovs).items()
                        for k, v in v0.items()
                    }
                ],
                schema=ovs_results.schema,
            ).with_columns(name=pl.lit(f"{name}_{sampling_strategy}"))
        )


**Задание 3. (1 балл)**. До этого момента все наши решения концентрировались на работе с минорным классом. Теперь давайте попробуем зайти с другой стороны. Может быть, для восстановления закономерностей нам не нужно столько объектов мажорного класса, и они просто засоряют нам выборку лишней информацией?

Для решения этой проблемы существуют методы андерсэмплинга. Самое простое, что можно придумать - удалять точки мажорного класса, пока мы не получим приемлемый баланс. Протестируйте следующий метод и постройте графики достигаемых значений метрик от баланса классов и от отношения размеров исходной и пересэмпленной выборки:

In [ ]:
from imblearn.under_sampling import RandomUnderSampler

In [ ]:
cbc_uns = cbc_ovs

In [ ]:
uns = RandomUnderSampler(
    sampling_strategy=0.5,
    random_state=132,
)
X_train_uns, y_train_uns = uns.fit_resample(
    df_train.to_numpy(), df_train["Class"].to_numpy()
)
X_train_uns = pl.DataFrame(X_train_uns, schema=df_train.columns)
y_train_uns = pl.Series("Class", y_train_uns)

train_pool_uns = Pool(X_train_uns.select(pl.selectors.starts_with("V")), y_train_uns)
X_train_uns["Class"].value_counts()

In [ ]:
cbc_uns.fit(train_pool_uns, eval_set=val_pool, verbose=False)

ovs_results = ovs_results.vstack(
    pl.DataFrame(
        [
            {
                f"{k0}_{k}": v
                for k0, v0 in print_stats(df_val, df_test, cbc_uns).items()
                for k, v in v0.items()
            }
        ],
        schema=ovs_results.schema,
    ).with_columns(name=pl.lit("random_under_sampler_0.5"))
)

In [ ]:
uns.set_params(sampling_strategy=0.1)
X_train_uns, y_train_uns = uns.fit_resample(
    df_train.to_numpy(), df_train["Class"].to_numpy()
)
X_train_uns = pl.DataFrame(X_train_uns, schema=df_train.columns)
y_train_uns = pl.Series("Class", y_train_uns)

train_pool_uns = Pool(X_train_uns.select(pl.selectors.starts_with("V")), y_train_uns)
X_train_uns["Class"].value_counts()

In [ ]:
cbc_uns.fit(train_pool_uns, eval_set=val_pool, verbose=False)

ovs_results = ovs_results.vstack(
    pl.DataFrame(
        [
            {
                f"{k0}_{k}": v
                for k0, v0 in print_stats(df_val, df_test, cbc_uns).items()
                for k, v in v0.items()
            }
        ],
        schema=ovs_results.schema,
    ).with_columns(name=pl.lit("random_under_sampler_0.1"))
)

Даже такой наивный подход может дать относительно неплохие результаты и улучшить наши метрики. Тем не менее, сейчас мы никак не используем информацию о распределении объектов в выборке. Оказывается, что даже относительно простые эвристические правила могут заметно поднять нам качество - например, мы можем при отбрасывании использовать близость отдельных объектов мажорного класса к минорному и отбрасывать самые близкие. Протестируйте алгоритм [Near-Miss](https://www.site.uottawa.ca/~nat/Workshop2003/jzhang.pdf) на наших данных и постройте графики, аналогичные предыдущему пункту (также добавьте график с зависимостью качества от числа соседей).

In [ ]:
from imblearn.under_sampling import NearMiss

undersample_dict = {
    "near_miss": NearMiss,
}

for name, undersampler in undersample_dict.items():
    uns = undersampler(
        # random_state=132,
    )
    for sampling_strategy in (0.5, 0.1):
        print(f"{name} with sampling_strategy={sampling_strategy}")
        uns.set_params(sampling_strategy=sampling_strategy)
        X_train_uns, y_train_uns = uns.fit_resample(
            df_train.to_numpy(), df_train["Class"].to_numpy()
        )
        X_train_uns = pl.DataFrame(X_train_uns, schema=df_train.columns)
        y_train_uns = pl.Series("Class", y_train_uns)

        train_pool_uns = Pool(
            X_train_uns.select(pl.selectors.starts_with("V")), y_train_uns
        )

        cbc_uns.fit(train_pool_uns, eval_set=val_pool, verbose=False)

        ovs_results = ovs_results.vstack(
            pl.DataFrame(
                [
                    {
                        f"{k0}_{k}": v
                        for k0, v0 in print_stats(df_val, df_test, cbc_uns).items()
                        for k, v in v0.items()
                    }
                ],
                schema=ovs_results.schema,
            ).with_columns(name=pl.lit(f"{name}_{sampling_strategy}"))
        )


**Бонус (1 балл)**. imblearn также предлагает много различных методов для андерсэмплинга. Выберите что-нибудь еще из предлагаемого на свой вкус, опишите идею метода и протестируйте его.


# Часть 2. Поиск аномалий. 

Как вы могли заметить, методы балансировки выборок очень часто могут привести к не самым лучшим результатам из-за того, что они по сути искажают информацию о реальном распределении данных - в реальности обычно требуется долгий подбор в принципе работоспособных для задачи методов и их аккуратная настройка. 

Теперь давайте попробуем слегка сменить постановку задачи и переключиться на задачу "одноклассовой" классификации, то есть - поиска *аномалий* в выборке. В общем-то, это вполне согласуется с нашей областью работы - мы действительно можем назвать мошеннические транзакции аномальньми (как интуитивно, так и на основании наблюдаемой балансировки данных). 

Стоит отметить, что методы обнаружения аномалий чаще всего относятся к классу методов обучения без учителя. Это дает некоторый положительный эффект - нам не обязательно нужно тратить время на разметку данных (тем не менее, для контроля качества какую-то часть разметить все-таки придется). Впрочем, чаще всего перфоманс таких методов оказывается заметно хуже, чем у честного обучения с учителем (если мы можем себе его позволить).



**Задание 4. (3 балла).** На занятиях вы разбирали схожий с Random Forest подход для детекции аномалий без известной разметки данных, называемый Isolation Forest. Напомним суть: на этапе обучения мы создаем ансамбль из решающих деревьев, в котором признак и порог на каждую вершину подбираются случайно. Затем мы считаем для объектов оценку аномальности через длину пути до соответствующего листа в каждом дереве.

В данном задании вам предлагается реализовать модификацию данного алгоритма, известную как Extended Isolation Forest. В ней мы на каждом шаге будем определять не порог для признака, а полноценную случайную гиперплоскость, разбивающую выборку на 2 части. С детальным описанием вы можете ознакомиться [здесь](https://arxiv.org/pdf/1811.02141.pdf).

Ниже приведен шаблон кода. Постарайтесь работать в его рамках (минорные изменения вполне допустимы, главное не переворачивайте всю структуру с ног на голову). 

**NB**: будем считать, что в нашем датасете нет категориальных признаков - можете не заморачиваться с их обработкой (но в общем случае, это будет важно).

**Советы**:
 - Численные признаки лучше предобработать надлежащим образом.
 - Возможно, вам поможет выбрасывание некоторых признаков.
 - Внимательно следите за знаками.
 - Не игнорируйте документирующие строки.
 - Вероятнее всего, вы не сможете приблизиться по качеству к supervised-решениям. Если у вас не получается это сделать (но вы уверены в своей правоте), не стоит тратить слишком много времени на поиск ошибок.
- При желании, можете вынести этот код в отдельный файл и приложить его к ноутбуку. Он довольно громоздкий и в ноутбуке такое плохо читается.


**Бонусы (каждый по 0.5)**:
- Сделайте ваш EIF параллельным (`multiprocessing`, `joblib`).
- Добавьте возможность откатиться к дефолтному варианту Isolation Forest. (порассуждайте, как можно реализовать это в данной модели?):


In [ ]:
def c_factor(n):
    """
    Computes average path length for an unsuccessful search in a binary search tree.
    Params:
        n: int - number of data points for BST
    """
    # your code here


def calc_height(X, depth, node):
    """
    Calculates anomaly scores for sample in a recursive manner.
    Params:
        X: np.array - current sample, available to node

        depth: int - path length up to current node

        node: Node - current tree node

    Returns:
        scores: int, float or np.array - anomaly scores for sample
    """
    scores = np.zeros(X.shape[0])

    # your code here


class Node(object):
    """
    A single node object for each tree. Contains information on height, current data,
    splitting hyperplane and children nodes.

    Attributes:
        X: np.array - data available to current node
        size: int - length of available data

        depth: int - depth of node

        left: Node - left child
        right: Node - right child

        kind: str - either "internal" or "external", indicates the type of current node

        w: np.array - normal vector for the splitting hyperplane
        b: float - intercept term for the splitting hyperplane
    """

    def __init__(self, X, depth, left, right, kind, w, b):
        """
        Node(h, left, right, kind, w, b)
        Represents the node object.

        Params:
            X: np.array - data available to current node
            depth: int - depth of node

            left: Node - left child
            right: Node - right child

            kind: str - either "internal" or "external", indicates the type of current node

            w: np.array - normal vector for the splitting hyperplane
            b: float - intercept term for the splitting hyperplane

        """
        self.size = len(X)

        self.depth = depth

        self.left = left
        self.right = right

        self.kind = kind

        self.w = w
        self.b = b

    def __repr__(self):
        """
        For convenience only.
        """
        return f"Node(size={self.size}, depth={self.depth}, kind={self.kind})"


class RandomizedTree(object):
    """
    Single randomized tree object. Stores root and its depth (tree is built recursively).
    Attributes:
        depth: int - current tree depth

        max_depth: int - maximum tree depth

        root: Node - root node

        internal_count: int - number of internal nodes

        external_count: int - number of external nodes

    """

    def __init__(self, X, max_depth):
        """
        Single randomized tree object. Stores root and its depth (tree is built recursively).
        Params:
            X: np.array - train sample
            max_depth: int - maximum tree depth

        """
        self.depth = 0
        self.max_depth = max_depth

        self.internal_count = 0
        self.external_count = 0

        self.root = self.grow(X, 0)

    def __repr__(self):
        """
        For convenience only.
        """

        return f"RandomizedTree(depth={self.depth}, max_depth={self.max_depth}, n_internal={self.internal_count}, n_external={self.external_count})"

    def grow(self, X, depth):
        """
        Grow tree in a recursive manner.
        Params:
            X: np.array - available train sample

            depth: int - current tree depth

        Returns:
            node: Node - a trained node with separating hyperplane data.
                         Node provides access to children if necessary (these are built recursively)
        """
        # your code here

        return Node(X, depth, left, right, kind, w, b)

    def score_samples(self, X):
        """
        Calculate anomaly scores for given data. You may utilize outer function `calc_height`.
        Params:
            X: np.array - data to be evaluated

        Returns:
            scores: np.array - estimated anomaly scores
        """
        # your code here

        return scores


class ExtendedIsolationForest(object):
    """
    Extended Isolation Forest object. Stores training data and trained randomized trees.
    Attributes:
        n_trees: int - number of Randomized Trees

        max_depth: int - maximum depth of each tree

        subsample_rate: float - draw `subsample_rate * X.shape[0]` samples for each tree

        trees: list - container for trained trees

        contamination: float - estimated fraction of anomaly samples in data. Used for thresholding

    """

    def __init__(self, n_trees, subsample_rate, max_depth=None, contamination=0.01):
        """
        Extended Isolation Forest object. Stores training data and trained randomized trees.
        Params:
            n_trees: int - number of Randomized Trees

            subsample_rate: float - draw `subsample_rate * X.shape[0]` samples for each tree

            max_depth: int or None - maximum depth of each tree. Defaults to ceil(log_2(subsample_size)) if not provided

            contamination: float - estimated fraction of anomaly samples in data. Used for thresholding

        """
        self.n_trees = n_trees
        self.max_depth = max_depth
        self.subsample_rate = subsample_rate
        self.trees = []
        self.contamination = contamination
        self.is_fit = False

    def __repr__(self):
        """For convenience only."""

        return f"ExtendedIsolationForest(n_trees={self.n_trees}, max_depth={self.max_depth}, subsample_rate={self.subsample_rate}, contamination={self.contamination}, is_fit={self.is_fit})"

    def fit(self, X):
        """
        Fit EIF to new data.
        Params:
            X: np.array - 2d array of samples
        """
        # your code here
        return self

    def score_samples(self, X):
        """
        Estimate (normalized) anomaly score for each given sample
        Params:
            X: np.array - new samples

        Returns:
            scores: np.array - anomaly scores (larger value means higher probability of a sample being an outlier)
        """
        # your code here

        return scores

    def predict(self, X):
        """
        Predict if given samples are outliers.
        Params:
            X: np.array - new samples

        Returns:
            labels: np.array - anomaly labels (1 for outliers, 0 for inliers)
        """
        # your code here

        return labels


**Задание 5. (1 балл).** Протестируйте вашу реализацию EIF и подберите оптимальные гиперпараметры (наш метод не использует разметку, поэтому можете попробовать делать это на обучающей выборке). Сравните ее с обычным IF из `sklearn` (желательно делать это на одних и тех же параметрах). Удалось ли сделать лучше?

In [ ]:
# your code here

Возможно, ваш алгоритм выдал большие оценки объектам с негативной разметкой. Постарайтесь выбрать несколько таких объектов и доступно объяснить (= с кодом и графиками), почему так вышло:

In [ ]:
# your code here

**Задание 6. (1 балл).** `sklearn` также предлагает нам и другие методы для поиска аномалий. В этом задании мы предлагаем вам сделать следующее:

Для начала попробуйте использовать методы Local Outlier Factor и One-Class SVM. Сравните результаты с IF и EIF.  

In [ ]:
# your code here

У вас началась депрессия из-за плохих метрик? Не беда! Сейчас давайте сделаем так: все методы, опробованные в этой части, попробуем задействовать для создания дополнительных признаков в данных. Проделайте это (не забудьте, что обучаться здесь нужно на трейне). Теперь возьмите лучшую модель из предыдущей части и обучите на новых данных. Смогли ли unsupervised-методы повысить вам качество?

In [ ]:
# your code here

**Бонус. (0.1 балла).**

При сдаче проверяющий запустит следующую клетку один раз. Если она даст положительный результат, вы получите 0.1 бонусных балла. Если она даст отрицательный результат, вы получите -0.1 бонусных балла. 

Если вы хотите отказаться от сдачи данного задания, допишите "хочу" после двоеточия: `your text here`

Ниже вы можете попрактиковаться и оценить ваши силы (изменять код ячейки запрещается!)

In [ ]:
import random

rng = random.SystemRandom(0)
rng.uniform(-1.0, 1.0)